# 04 · Regular Expressions (`re`)

**Module 12 – Standard Library Essentials** · Estimated time: 70 min

> Merges 📗 *Overview of Regular Expressions* (very thorough) with 📘 *Regular Expressions in Python*, and adds named groups, greedy vs lazy, flags, `sub()` with functions, lookarounds, `fullmatch` and validation recipes.

### 🎯 What you will learn
- What regex is; why we use raw strings `r"..."`
- `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub`, `split`, `compile`
- Character classes (`\d \w \s` ...), quantifiers (`+ * ? {m,n}`)
- Groups, named groups, alternation `|`
- Anchors `^ $ \b`, exclusion `[^...]`, wildcard `.`
- Greedy vs lazy matching; flags
- Real-world recipes: emails, phone numbers, dates, log parsing

## 1. Searching for basic patterns (📗)

A **regular expression** is a mini-language for describing text patterns, e.g. "three digits, a dash, four digits".

In [1]:
import re

text = "The agent's phone number is 408-555-1234. Call soon!"
print("phone" in text)            # plain substring check

match = re.search("phone", text)  # returns a Match object, or None
print(match)
print(match.span(), match.start(), match.end(), match.group())

True
<re.Match object; span=(12, 17), match='phone'>
(12, 17) 12 17 phone


In [2]:
text = "my phone is a new phone"
print(re.search("phone", text).span())       # only the FIRST match
print(re.findall("phone", text))             # all matches as strings
print(len(re.findall("phone", text)))
for m in re.finditer("phone", text):         # all matches as Match objects
    print(m.span(), m.group())

(3, 8)
['phone', 'phone']
2
(3, 8) phone
(18, 23) phone


### 📘 Handbook example

In [3]:
text = "The quick brown fox jumps over the lazy dog."
match = re.search("brown", text)
if match:
    print("Match found! Start index:", match.start(), "End index:", match.end())
print("Matches:", re.findall("the", text, re.IGNORECASE))    # case-insensitive
print("New text:", re.sub("fox", "cat", text))               # replace

Match found! Start index: 10 End index: 15
Matches: ['The', 'the']
New text: The quick brown cat jumps over the lazy dog.


## 2. Patterns: character identifiers (📗)

| Character | Description | Example pattern | Example match |
|---|---|---|---|
| `\d` | a digit | `file_\d\d` | `file_25` |
| `\w` | alphanumeric or `_` | `\w-\w\w\w` | `A-b_1` |
| `\s` | whitespace | `a\sb\sc` | `a b c` |
| `\D` | a non-digit | `\D\D\D` | `ABC` |
| `\W` | non-alphanumeric | `\W\W\W\W\W` | `*-+=)` |
| `\S` | non-whitespace | `Yoyo\S` | `Yoyo!` |
| `.` | any character except newline | `a.c` | `abc`, `a-c` |

> ✅ **Always use raw strings** (`r"\d"`) for regex patterns, so Python doesn't interpret the backslashes first.

In [4]:
text = "My telephone number is 408-555-1234"
phone = re.search(r"\d\d\d-\d\d\d-\d\d\d\d", text)
print(phone.group())

408-555-1234


## 3. Quantifiers (📗)

| Quantifier | Meaning | Example | Matches |
|---|---|---|---|
| `+` | 1 or more | `Version \w-\w+` | `Version A-b1_1` |
| `{3}` | exactly 3 | `\D{3}` | `abc` |
| `{2,4}` | 2 to 4 | `\d{2,4}` | `123` |
| `{3,}` | 3 or more | `\w{3,}` | `anycharacters` |
| `*` | 0 or more | `A*B*C*` | `AAACC` |
| `?` | 0 or 1 (optional) | `plurals?` | `plural`, `plurals` |

In [5]:
print(re.search(r"\d{3}-\d{3}-\d{4}", text).group())
print(re.findall(r"colou?r", "color colour colr"))
print(re.findall(r"\d+", "Order 66 shipped 3 items for 1250 rupees"))

408-555-1234
['color', 'colour']
['66', '3', '1250']


## 4. Groups `( )` (📗)

Parentheses capture parts of a match separately.

In [6]:
phone_pattern = re.compile(r"(\d{3})-(\d{3})-(\d{4})")   # compile once, reuse many times
results = re.search(phone_pattern, text)
print(results.group())      # the whole match
print(results.group(1))     # first group (area code)
print(results.group(2), results.group(3))
print(results.groups())

408-555-1234
408
555 1234
('408', '555', '1234')


### Named groups: `(?P<name>...)`

In [7]:
m = re.search(r"(?P<area>\d{3})-(?P<exchange>\d{3})-(?P<line>\d{4})", text)
print(m["area"], m.group("line"))
print(m.groupdict())

408 1234
{'area': '408', 'exchange': '555', 'line': '1234'}


With multiple groups, `findall` returns **tuples**:

In [8]:
print(re.findall(r"(\w+)@(\w+)\.com", "a@x.com, bob@site.com"))

[('a', 'x'), ('bob', 'site')]


## 5. Additional regex syntax (📗)

### OR operator `|`

In [9]:
print(re.search(r"man|woman", "This man was here."))
print(re.search(r"man|woman", "This woman was here."))

<re.Match object; span=(5, 8), match='man'>
<re.Match object; span=(5, 10), match='woman'>


### The wildcard `.`

In [10]:
print(re.findall(r".at", "The cat in the hat sat here."))
print(re.findall(r"...at", "The bat went splat"))
print(re.findall(r"\S+at", "The bat went splat"))     # one or more non-whitespace + "at"

['cat', 'hat', 'sat']
['e bat', 'splat']
['bat', 'splat']


### Starts with `^` and ends with `$`

In [11]:
print(re.findall(r"\d$", "This ends with a number 2"))     # ends with a digit
print(re.findall(r"^\d", "1 is the loneliest number."))    # starts with a digit

['2']
['1']


### Exclusion `[^...]`

In [12]:
phrase = "there are 3 numbers 34 inside 5 this sentence."
print(re.findall(r"[^\d]", phrase)[:12])
print(re.findall(r"[^\d]+", phrase))       # runs of non-digits

test_phrase = "This is a string! But it has punctuation. How can we remove it?"
clean = re.findall(r"[^!.? ]+", test_phrase)
print(" ".join(clean))

['t', 'h', 'e', 'r', 'e', ' ', 'a', 'r', 'e', ' ', ' ', 'n']
['there are ', ' numbers ', ' inside ', ' this sentence.']
This is a string But it has punctuation How can we remove it


### Brackets for grouping characters `[ ]`

In [13]:
text = "Only find the hypen-words in this sentence. But you do not know how long-ish they are"
print(re.findall(r"[\w]+-[\w]+", text))
print(re.findall(r"[aeiou]{2}", "beautiful queue"))       # two vowels in a row
print(re.findall(r"[A-Z][a-z]+", "Asha met Ravi in Pune"))

['hypen-words', 'long-ish']
['ea', 'ue', 'ue']
['Asha', 'Ravi', 'Pune']


### Parentheses for multiple options (📗)

In [14]:
for t in ["Hello, would you like some catfish?", "Hello, would you like to take a catnap?",
          "Hello, have you seen this caterpillar?"]:
    print(re.search(r"cat(fish|nap|claw)", t))

<re.Match object; span=(27, 34), match='catfish'>
<re.Match object; span=(32, 38), match='catnap'>
None


## 6. Word boundaries `\b` (📘)

In [15]:
text = "The quick brown fox jumps over the lazy dog."
pattern = re.compile(r"\b\w+\b")        # matches whole words
print("Words:", pattern.findall(text))
print(re.findall(r"\bcat\b", "cat concat cat's category"))

Words: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
['cat', 'cat']


| Part | Meaning |
|---|---|
| `\b` | word boundary (start/end of a word) |
| `\w+` | one or more word characters (letters, digits, underscores) |
| `\b` | word boundary |

## 7. `match`, `fullmatch` vs `search`

| Function | Where it looks |
|---|---|
| `re.search(p, s)` | anywhere in the string |
| `re.match(p, s)` | only at the **beginning** |
| `re.fullmatch(p, s)` | the **entire** string must match: ✅ use this for validation |

In [16]:
print(re.search(r"\d+", "abc 123"), re.match(r"\d+", "abc 123"), re.match(r"\d+", "123 abc"))
print(bool(re.fullmatch(r"\d{6}", "411001")), bool(re.fullmatch(r"\d{6}", "411001x")))

<re.Match object; span=(4, 7), match='123'> None <re.Match object; span=(0, 3), match='123'>
True False


> 🆕 **Python 3.15:** `re.match()` is *soft-deprecated* in favour of the clearer name **`re.prefixmatch()`** (same behaviour). `re.match` won't be removed; new code may prefer `prefixmatch` when targeting 3.15+.

In [17]:
print(re.prefixmatch(r"\d+", "123 abc"))

<re.Match object; span=(0, 3), match='123'>


## 8. Greedy vs lazy quantifiers

By default `*`, `+`, `?` are **greedy** (match as much as possible). Add `?` to make them **lazy** (as little as possible):

In [18]:
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.*>", html))      # greedy: one giant match
print(re.findall(r"<.*?>", html))     # lazy: each tag

['<b>bold</b> and <i>italic</i>']
['<b>', '</b>', '<i>', '</i>']


## 9. Flags

In [19]:
text = "Python\npython\nPYTHON"
print(re.findall(r"^python$", text, flags=re.IGNORECASE | re.MULTILINE))
pattern = re.compile(r"""
    (?P<user>[\w.+-]+)     # user name
    @
    (?P<domain>[\w-]+\.[\w.]+)   # domain
""", re.VERBOSE)               # VERBOSE allows whitespace and comments in patterns
print(pattern.search("contact: asha.k+news@mail.example.co.in").groupdict())

['Python', 'python', 'PYTHON']
{'user': 'asha.k+news', 'domain': 'mail.example.co.in'}


| Flag | Effect |
|---|---|
| `re.I` / `IGNORECASE` | case-insensitive |
| `re.M` / `MULTILINE` | `^` `$` match at every line |
| `re.S` / `DOTALL` | `.` also matches newline |
| `re.X` / `VERBOSE` | allow whitespace & comments |

## 10. Substitution and splitting

In [20]:
print(re.sub(r"\s+", " ", "too    many     spaces"))
print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", "Due 2026-10-20"))   # back-references
print(re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "3 apples, 10 pears"))   # function!
print(re.split(r"[,;\s]+", "a, b;c  d"))
print(re.subn(r"o", "0", "foo boo"))     # also returns the number of replacements

too many spaces
Due 20/10/2026
6 apples, 20 pears
['a', 'b', 'c', 'd']
('f00 b00', 4)


## 11. Lookarounds (advanced)

Assert that something is (or isn't) next to the match, without including it:

In [21]:
print(re.findall(r"\d+(?=₹)", "Cost: 500₹ or 20$"))           # lookahead: digits followed by ₹
print(re.findall(r"(?<=\$)\d+", "Price $45 and $120"))        # lookbehind: digits preceded by $
password = "Secr3tPass"
strong = re.fullmatch(r"(?=.*[A-Z])(?=.*[a-z])(?=.*\d).{8,}", password)
print("strong password?", bool(strong))

['500']
['45', '120']
strong password? True


## 12. Real-world recipes

In [22]:
log = """2026-10-20 10:15:32 ERROR [payment] Timeout after 30s user=asha
2026-10-20 10:16:01 INFO [auth] Login ok user=ravi
2026-10-20 10:17:45 ERROR [upload] File too large user=meena"""

line_re = re.compile(r"^(?P<date>\S+) (?P<time>\S+) (?P<level>\w+) \[(?P<module>\w+)\] (?P<msg>.*?) user=(?P<user>\w+)$",
                     re.MULTILINE)
for m in line_re.finditer(log):
    if m["level"] == "ERROR":
        print(m["time"], m["module"], "-", m["msg"], f"({m['user']})")

10:15:32 payment - Timeout after 30s (asha)
10:17:45 upload - File too large (meena)


In [23]:
text = "Contact asha@example.com or support@help.co.in. Call +91 98765 43210 or 020-2567-8901 by 20/10/2026."
print(re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+\w", text))                  # emails (simplified)
print(re.findall(r"\+?\d[\d\s-]{8,}\d", text))                         # phone-like numbers
print(re.findall(r"\b\d{2}/\d{2}/\d{4}\b", text))                       # dates dd/mm/yyyy
print(bool(re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", "ABCDE1234F")))          # PAN-card format

['asha@example.com', 'support@help.co.in']
['+91 98765 43210', '020-2567-8901']
['20/10/2026']
True


> 💡 Real email validation is very complex; for production use a library (e.g. `email-validator`) or simply send a confirmation email. Test patterns interactively at https://regex101.com (choose the *Python* flavour).

## 📝 Summary

| Function | Returns |
|---|---|
| `search` | first match anywhere (or `None`) |
| `match` / 🆕 `prefixmatch` | match at the start |
| `fullmatch` | whole string must match |
| `findall` | list of strings (or tuples of groups) |
| `finditer` | iterator of Match objects |
| `sub` / `subn` | replaced string (/ + count) |
| `split` | list split by the pattern |
| `compile` | reusable Pattern object |

## 🏋️ Practice (📗 Advanced Modules Puzzle uses this, see Module 11 exercises)
1. Extract all hashtags from `"Loving #Python and #regex! #100DaysOfCode"`.
2. Validate Indian PIN codes (exactly 6 digits, not starting with 0).
3. Convert `"camelCaseString"` to `"camel_case_string"`.
4. Find all words of 5+ letters in a sentence.
5. Mask all but the last 4 digits of a card number `"4111 1111 1111 1234"`.

<details><summary>Show solutions</summary>

```python
import re
print(re.findall(r"#\w+", "Loving #Python and #regex! #100DaysOfCode"))
print(bool(re.fullmatch(r"[1-9]\d{5}", "411001")))
print(re.sub(r"(?<!^)(?=[A-Z])", "_", "camelCaseString").lower())
print(re.findall(r"\b[a-zA-Z]{5,}\b", "Regular expressions are really useful"))
print(re.sub(r"\d(?=(?:[\d ]*\d){4})", "*", "4111 1111 1111 1234"))
```
</details>